# 01 tokenizer 到底做了什么：文字到数字

今天只解决一个问题：

> 为什么模型不能直接读中文？tokenizer 到底把 prompt 变成了什么？

大模型真正吃的是整数 token id，不是 Python 字符串。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, textwrap

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path(r"D:/coding/qwen lorar sft/qwen-local-lora-sft-dpo")

print("项目根目录:", PROJECT_ROOT)
print("本 micro notebook 默认只读代码；真实模型推理开关默认 False。")

def read_text(rel):
    return (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace")

def show_file(rel, start=1, end=80):
    lines = read_text(rel).splitlines()
    print(f"--- {rel}:{start}-{min(end, len(lines))} ---")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03d}: {lines[i-1]}")

def find_lines(rel, keyword, context=4):
    lines = read_text(rel).splitlines()
    hits = [i for i, line in enumerate(lines, start=1) if keyword in line]
    if not hits:
        print("没有找到:", keyword)
        return
    for hit in hits:
        show_file(rel, max(1, hit-context), min(len(lines), hit+context))
        print()

## 1. tokenizer 在 infer.py 里从哪里来？

In [ ]:
find_lines("scripts/infer.py", "AutoTokenizer.from_pretrained", context=8)

这行代码会根据模型名 `Qwen/Qwen2.5-0.5B-Instruct` 加载匹配的 tokenizer。

如果本地 `.hf_cache` 已经有缓存，`local_files_only=True` 就只从本地读。

## 2. chat template 是什么？

聊天模型不只是看一句 prompt，它还要知道哪个是 system，哪个是 user，哪里轮到 assistant 回答。项目用 `tokenizer.apply_chat_template(...)`，而不是手写字符串拼接。

In [ ]:
find_lines("scripts/infer.py", "messages = [", context=10)
find_lines("scripts/infer.py", "apply_chat_template", context=12)

## 3. token ids 长什么样？

下面这格默认不加载模型。你确认环境和缓存以后，可以把 `RUN_TOKENIZER_DEMO=True` 看真实 token ids。

In [ ]:
RUN_TOKENIZER_DEMO = False
if RUN_TOKENIZER_DEMO:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", local_files_only=True)
    messages = [
        {"role": "system", "content": "你是一个清晰严谨的中文助手。"},
        {"role": "user", "content": "请解释 LoRA。"},
    ]
    ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
    print("token 数量:", len(ids))
    print("前 30 个 token id:", ids[:30])
    print(tokenizer.decode(ids[:80]))
else:
    print("默认不加载 tokenizer。你理解后可改 True。")

## 4. 和后面 SFT 的关系

SFT 训练里也用同一个 tokenizer。区别是：推理只有 system + user，后面让模型生成 assistant；SFT 有 system + user + assistant 标准答案，用来算 loss。

In [ ]:
find_lines("scripts/train_sft_lora.py", "prompt_ids =", context=10)
find_lines("scripts/train_sft_lora.py", "full_ids =", context=10)

## 5. 本节面试三句话

1. tokenizer 是文字到 token id 的转换器。
2. chat template 负责把 system/user/assistant 包成 Qwen 熟悉的聊天格式。
3. 推理和训练都依赖同一个 tokenizer，否则输入格式会乱。